In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class OMethylation(MorphingOperator):
    def __init__(self):
        super(OMethylation, self).__init__()
        self._name = "O-methylation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")

    def setOriginal(self, mol):
        super(OMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_o_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_o_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

omethylation_op = OMethylation() 

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target


start_mol = MolpherMol("CC(=O)Nc1ccc(O)cc1") # Παρακεταμόλη (Φαινόλη)
target_mol = MolpherMol("CC(=O)Nc1ccc(OC)cc1") # O-Μεθυλιωμένη Παρακεταμόλη (Στόχος)
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (omethylation_op,)

closest_info = FindClosest()

print("--- STARTING O-METHYLATION TREE SEARCH ---")
while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
        
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
        
    if tree.path_found or tree.generation_count >= 5:
        break

print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS: Target reached! The tree completed the O-Methylation.")


print("\n=== RUNNING FALSE POSITIVE TRAP TESTS ===")
traps = {
        "Carboxylic Acid Trap (CC(=O)O - Όχι αλκοόλη)": "CC(=O)O",
        "Enol Trap (C=CO - Ασταθής ενόλη)": "C=CO",
        "Hydroperoxide Trap (CCOO - Υπεροξείδιο)": "CCOO",
        "Sulfonic Acid Trap (CCS(=O)(=O)O)": "CCS(=O)(=O)O"
}
    
for name, smiles in traps.items():
    mol = MolpherMol(smiles)
    omethylation_op.setOriginal(mol)
    product = omethylation_op.morph()
        
    safe = (product.getSMILES() == mol.getSMILES())
    print(f"{name}:")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  STATUS: {'SAFE (Passed)' if safe else 'VULNERABLE (Failed)'}")
    print("-" * 50)

--- STARTING O-METHYLATION TREE SEARCH ---
Generation #1
Molecules in tree: 2
Closest to target: COC1=CC=C(NC(C)=O)C=C1 (Distance: 0.0000)
----------------------------------------

Search finished!
SUCCESS: Target reached! The tree completed the O-Methylation.

=== RUNNING FALSE POSITIVE TRAP TESTS ===
Carboxylic Acid Trap (CC(=O)O - Όχι αλκοόλη):
  SOURCE: CC(=O)O
  STATUS: SAFE (Passed)
--------------------------------------------------
Enol Trap (C=CO - Ασταθής ενόλη):
  SOURCE: C=CO
  STATUS: SAFE (Passed)
--------------------------------------------------
Hydroperoxide Trap (CCOO - Υπεροξείδιο):
  SOURCE: CCOO
  STATUS: SAFE (Passed)
--------------------------------------------------
Sulfonic Acid Trap (CCS(=O)(=O)O):
  SOURCE: CCS(=O)(=O)O
  STATUS: SAFE (Passed)
--------------------------------------------------
